# Merge

In [3]:
from pathlib import Path

import rasterio
from rasterio.merge import merge


# -------------------------------------------------------------------
# User settings
# -------------------------------------------------------------------
input_dir = Path(
    r"C:\Users\PangY\OneDrive - Smithsonian Institution\Bustard"
    r"\01_Data\GEE Satellite\GEE_S2_Export\S2_Veg_Pred_Seasonal"
)

# Finds all chunks belonging to this prediction.
tile_pattern = "S2_RF_EM_2017_JJA-*.tif"

output_file = input_dir / "S2_RF_EM_2017_JJA_merged.tif"


# -------------------------------------------------------------------
# Find input tiles
# -------------------------------------------------------------------
tile_paths = sorted(input_dir.glob(tile_pattern))

if not tile_paths:
    raise FileNotFoundError(
        f"No files were found using:\n{input_dir / tile_pattern}"
    )

print(f"Found {len(tile_paths)} tiles:")
for path in tile_paths:
    print("  ", path.name)


# -------------------------------------------------------------------
# Open and validate tiles
# -------------------------------------------------------------------
sources = [rasterio.open(path) for path in tile_paths]

try:
    reference = sources[0]

    for source in sources[1:]:
        if source.crs != reference.crs:
            raise ValueError(
                f"CRS mismatch:\n"
                f"{reference.name}: {reference.crs}\n"
                f"{source.name}: {source.crs}"
            )

        if source.res != reference.res:
            raise ValueError(
                f"Resolution mismatch:\n"
                f"{reference.name}: {reference.res}\n"
                f"{source.name}: {source.res}"
            )

        if source.count != reference.count:
            raise ValueError("The tiles have different numbers of bands.")

    # Merge adjacent tiles.
    nodata_value = -9999.0
    mosaic, mosaic_transform = merge(
        sources,
        method="first",
        nodata=nodata_value
    )

    # Copy metadata from the first tile.
    output_metadata = reference.meta.copy()

    output_metadata.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": mosaic_transform,
        "count": mosaic.shape[0],
        "compress": "deflate",
        "tiled": True,
        "bigtiff": "IF_SAFER"
    })

    # Write the merged raster.
    with rasterio.open(
        output_file,
        "w",
        **output_metadata
    ) as destination:
        destination.write(mosaic)

finally:
    for source in sources:
        source.close()


print(f"\nMerged raster saved to:\n{output_file}")

Found 2 tiles:
   S2_RF_EM_2017_JJA-0000000000-0000000000.tif
   S2_RF_EM_2017_JJA-0000000000-0000032768.tif

Merged raster saved to:
C:\Users\PangY\OneDrive - Smithsonian Institution\Bustard\01_Data\GEE Satellite\GEE_S2_Export\S2_Veg_Pred_Seasonal\S2_RF_EM_2017_JJA_merged.tif
